# 09 - Dinamik TOPSIS ile 2025-2030 İl Bazlı Bütçe Dağıtımı

Bu notebook, klasik TOPSIS listesindeki sabit sıralama problemini çözmek için yıl yıl geri beslemeli çalışır.

**Mantık:**
1. Her yıl için illerin TOPSIS önem sırası hesaplanır.
2. O yılın toplam sağlık bütçesi, TOPSIS skorlarından türetilen ihtiyaç payına göre illere dağıtılır.
3. Dağıtılan bütçe, sonraki yılın TOPSIS hesabına **önceki kümülatif bütçe desteği** olarak girer.
4. Önceki yıllarda yüksek destek alan illerin sonraki yıllarda tekrar aynı şekilde üst sırada kalması engellenir.

**Not:** DSÖ sağlık finansmanını sağlık sisteminin temel fonksiyonlarından biri olarak görür; ancak harcama tek başına sağlık çıktısını açıklamaz. Bu nedenle bütçe etkisi varsayılan olarak TOPSIS içinde %25 ağırlıkla modellenmiştir. Bu oran `BUDGET_EFFECT_WEIGHT` değişkeninden kolayca değiştirilebilir.

In [2]:
# %% [1] KÜTÜPHANELER VE DOSYA YOLLARI
import os
import re
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


# Girdi dosyaları
TOPSIS_VERI_PATH = "2025_2030_tum_hedefler_final_tahmin.csv"
ENSEMBLE_PATH    = "tahmin_2025_2030_ensemble_sabit.csv"
BUDGET_PATH      = "butce_2025_2030_tahmin_stabil.csv"

# Analiz yılları
GELECEK_YILLAR = list(range(2025, 2031))

# Bütçe dağıtımında kullanılacak bütçe kolonu
# TL bazlı dağıtım için:
BUDGET_VALUE_COL = "butce_TL_turetilen (Milyon TL)"

# Alternatif olarak USD bazlı dağıtım yapmak istersen:
# BUDGET_VALUE_COL = "butce_USD_tahmin (Milyon USD)"

In [3]:
# %% [2] MODEL PARAMETRELERİ
# ---------------------------------------------------------------------
# BÜTÇE ETKİ AĞIRLIĞI
# ---------------------------------------------------------------------
# DSÖ sağlık finansmanını sağlık sisteminin temel fonksiyonu olarak tanımlar.
# Fakat harcama düzeyi tek başına sağlık çıktısını garanti etmez.
# Bu yüzden bütçeyi TOPSIS'te güçlü ama baskın olmayan bir etki olarak alıyoruz.
#
# 0.20 -> daha zayıf bütçe geri beslemesi
# 0.25 -> önerilen / dengeli varsayılan
# 0.30 -> daha güçlü bütçe geri beslemesi
BUDGET_EFFECT_WEIGHT = 0.25

# ---------------------------------------------------------------------
# BÜTÇE DESTEĞİ NASIL TOPSIS'E GİRSİN?
# ---------------------------------------------------------------------
# "per_capita": önceki yıllarda alınan kümülatif bütçe / nüfus
#               Küçük-büyük şehirleri daha adil karşılaştırır. Önerilen.
# "absolute":   önceki yıllarda alınan toplam kümülatif bütçe
#               Büyük bütçe alan ili doğrudan daha güçlü cezalandırır.
BUDGET_SUPPORT_MODE = "per_capita"

# ---------------------------------------------------------------------
# BÜTÇE DAĞITIM YOĞUNLUĞU
# ---------------------------------------------------------------------
# 1.00 -> bütçe doğrudan TOPSIS skor payına göre dağılır.
# 1.20 -> üst sıralara biraz daha fazla ağırlık verir.
# 0.80 -> dağılımı biraz daha dengeler.
BUDGET_ALLOCATION_GAMMA = 1.0

# ---------------------------------------------------------------------
# İlk yılın başında önceki destek yoktur.
# 2025 TOPSIS hesabında budget_support = 0 olur.
# 2025 bütçesi dağıtıldıktan sonra 2026 hesabına girer.
# ---------------------------------------------------------------------
FIRST_YEAR_SUPPORT_ZERO = True

In [4]:
# %% [3] TOPSIS KRİTERLERİ VE TİPLERİ
# benefit: yüksek değer = yüksek ihtiyaç / yüksek öncelik
# cost:    yüksek değer = düşük ihtiyaç / düşük öncelik

BASE_CRITERIA_TYPES = {
    "oran_65_plus":  "benefit",   # 65+ oranı arttıkça sağlık ihtiyacı artar
    "oran_0_14":     "benefit",   # 0-14 oranı arttıkça sağlık ihtiyacı artar
    "hekim_1000":    "cost",      # 1000 kişiye düşen hekim arttıkça ihtiyaç azalır
    "yatak_100000":  "cost",      # 100 bin kişiye düşen yatak arttıkça ihtiyaç azalır
}

BASE_CRITERIA_COLS = list(BASE_CRITERIA_TYPES.keys())

# Dinamik kriter: önceki yıllarda alınan kümülatif bütçe desteği
# Yüksek destek almış il, sonraki yıl daha düşük öncelik almalı.
BUDGET_SUPPORT_COL = "onceki_kumulatif_butce_destegi"

In [5]:
# %% [4] YARDIMCI FONKSİYONLAR
TR_MAP = str.maketrans({
    "İ": "i", "I": "i", "ı": "i",
    "Ş": "s", "ş": "s",
    "Ğ": "g", "ğ": "g",
    "Ü": "u", "ü": "u",
    "Ö": "o", "ö": "o",
    "Ç": "c", "ç": "c",
})

def normalize_city_name(x):
    """İl adlarını merge için standart anahtara çevirir."""
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"-\d+$", "", x)
    x = x.translate(TR_MAP).lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def assign_priority_group(rank):
    """81 il için yaklaşık öncelik grubu."""
    if rank <= 20:
        return "Çok yüksek öncelik"
    elif rank <= 40:
        return "Yüksek öncelik"
    elif rank <= 60:
        return "Orta öncelik"
    else:
        return "Düşük öncelik"

def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")

def check_file_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dosya bulunamadı: {path}")

def normalize_to_sum_one(values):
    """Negatif olmayan değerleri toplamı 1 olacak şekilde normalize eder."""
    values = np.asarray(values, dtype=float)
    values = np.where(np.isfinite(values), values, 0.0)
    values = np.clip(values, 0, None)
    total = values.sum()
    if np.isclose(total, 0):
        return np.ones(len(values)) / len(values)
    return values / total

In [6]:
# %% [5] SHANNON ENTROPİ AĞIRLIK FONKSİYONU
def shannon_entropi_agirlik(df, kriterler):
    """
    Shannon entropisi ile veriden objektif ağırlık hesaplar.

    1) Her kriter kolonu normalize edilir.
    2) Entropi değeri hesaplanır.
    3) Bilgi içeriği yüksek olan kriter daha yüksek ağırlık alır.
    """
    X = df[kriterler].values.astype(float)
    X = np.where(np.isfinite(X), X, np.nan)

    # Eksikleri medyan ile doldur
    for j in range(X.shape[1]):
        col = X[:, j]
        med = np.nanmedian(col)
        if np.isnan(med):
            med = 0.0
        col = np.where(np.isnan(col), med, col)
        X[:, j] = col

    # Entropi için sıfır/negatif değerleri küçük pozitif sayıya çek
    X = np.where(X <= 0, 1e-9, X)

    X_norm = X / X.sum(axis=0, keepdims=True)
    k = 1.0 / np.log(len(df))
    E = -k * np.sum(X_norm * np.log(X_norm), axis=0)
    d = 1.0 - E

    if np.isclose(d.sum(), 0):
        w = np.ones(len(kriterler)) / len(kriterler)
    else:
        w = d / d.sum()

    return dict(zip(kriterler, w))

In [7]:
# %% [6] TOPSIS FONKSİYONU
def calculate_topsis_for_one_year(df_year, criteria_cols, weights, criteria_types):
    """Tek bir yıl için iller arası TOPSIS skoru hesaplar."""
    d = df_year.copy().reset_index(drop=True)

    # Eksik değerleri yıl medyanı ile doldur
    for col in criteria_cols:
        d[col] = safe_numeric(d[col])
        med = d[col].median()
        if pd.isna(med):
            med = 0.0
        d[col] = d[col].fillna(med)

    X = d[criteria_cols].astype(float).values

    # 1) Vektör normalizasyonu
    denom = np.sqrt((X ** 2).sum(axis=0))
    denom = np.where(denom == 0, 1e-12, denom)
    X_norm = X / denom

    # 2) Ağırlıklandırma
    weight_vector = np.array([weights[col] for col in criteria_cols], dtype=float)
    X_weighted = X_norm * weight_vector

    for idx, col in enumerate(criteria_cols):
        d[f"{col}_norm"]      = X_norm[:, idx]
        d[f"{col}_agirlikli"] = X_weighted[:, idx]

    # 3) İdeal iyi / ideal kötü
    ideal_best, ideal_worst = [], []

    for idx, col in enumerate(criteria_cols):
        vals = X_weighted[:, idx]

        if criteria_types[col] == "benefit":
            ideal_best.append(vals.max())
            ideal_worst.append(vals.min())
        elif criteria_types[col] == "cost":
            ideal_best.append(vals.min())
            ideal_worst.append(vals.max())
        else:
            raise ValueError(f"Bilinmeyen kriter tipi: {col}")

    ideal_best  = np.array(ideal_best)
    ideal_worst = np.array(ideal_worst)

    # 4) Uzaklıklar
    d_plus  = np.sqrt(((X_weighted - ideal_best)  ** 2).sum(axis=1))
    d_minus = np.sqrt(((X_weighted - ideal_worst) ** 2).sum(axis=1))

    # 5) TOPSIS skoru
    topsis_score = d_minus / (d_plus + d_minus + 1e-12)

    d["topsis_skor"]        = topsis_score
    d["ideal_iyi_uzaklik"]  = d_plus
    d["ideal_kotu_uzaklik"] = d_minus

    # 6) Sıralama + öncelik grubu
    d["topsis_sira"]   = d["topsis_skor"].rank(ascending=False, method="dense").astype(int)
    d["oncelik_grubu"] = d["topsis_sira"].apply(assign_priority_group)

    return d.sort_values("topsis_sira").reset_index(drop=True)

In [8]:
# %% [7] VERİLERİ OKU VE HAZIRLA
for p in [TOPSIS_VERI_PATH, ENSEMBLE_PATH, BUDGET_PATH]:
    check_file_exists(p)

topsis_raw = pd.read_csv(TOPSIS_VERI_PATH)
ensemble   = pd.read_csv(ENSEMBLE_PATH)
budget_df  = pd.read_csv(BUDGET_PATH)

topsis_raw.columns = [c.strip() for c in topsis_raw.columns]
ensemble.columns   = [c.strip() for c in ensemble.columns]
budget_df.columns  = [c.strip() for c in budget_df.columns]

# Yıl dönüşümleri
topsis_raw["yıl"] = safe_numeric(topsis_raw["yıl"]).astype(int)
ensemble["yıl"]   = safe_numeric(ensemble["yıl"]).astype(int)
budget_df["yıl"]  = safe_numeric(budget_df["yıl"]).astype(int)

# İl anahtarı
if "il_key" not in topsis_raw.columns:
    topsis_raw["il_key"] = topsis_raw["il_adı"].apply(normalize_city_name)
if "il_key" not in ensemble.columns:
    ensemble["il_key"] = ensemble["il_adı"].apply(normalize_city_name)

# Yıl filtresi
topsis_raw = topsis_raw[topsis_raw["yıl"].isin(GELECEK_YILLAR)].copy()
ensemble   = ensemble[ensemble["yıl"].isin(GELECEK_YILLAR)].copy()
budget_df  = budget_df[budget_df["yıl"].isin(GELECEK_YILLAR)].copy()

if BUDGET_VALUE_COL not in budget_df.columns:
    raise ValueError(
        f"Bütçe dosyasında '{BUDGET_VALUE_COL}' kolonu yok. "
        f"Mevcut kolonlar: {budget_df.columns.tolist()}"
    )

budget_df[BUDGET_VALUE_COL] = safe_numeric(budget_df[BUDGET_VALUE_COL])

print("TOPSIS ham veri:", topsis_raw.shape)
print("Ensemble veri:", ensemble.shape)
print("Bütçe veri:", budget_df.shape)
print("Bütçe kolonu:", BUDGET_VALUE_COL)

TOPSIS ham veri: (486, 14)
Ensemble veri: (486, 10)
Bütçe veri: (6, 6)
Bütçe kolonu: butce_TL_turetilen (Milyon TL)


In [9]:
# %% [8] TOPSIS KRİTER MATRİSİNİ OLUŞTUR
# 2025_2030_tum_hedefler_final_tahmin.csv içindeki kolonları TOPSIS formatına çeviriyoruz.
rename_map = {
    "yas_65_plus":  "oran_65_plus",
    "yas_0_14":     "oran_0_14",
    "hekim_oran":   "hekim_1000",
    "yatak_oran":   "yatak_100000",
}

topsis_criteria = topsis_raw.rename(columns=rename_map).copy()

required_topsis_cols = [
    "il_adı", "il_key", "yıl", "nufus",
    "oran_65_plus", "oran_0_14", "hekim_1000", "yatak_100000"
]

missing_cols = [c for c in required_topsis_cols if c not in topsis_criteria.columns]
if missing_cols:
    raise ValueError(f"TOPSIS kriter veri setinde eksik kolonlar var: {missing_cols}")

topsis_criteria = topsis_criteria[required_topsis_cols].copy()

for col in ["nufus"] + BASE_CRITERIA_COLS:
    topsis_criteria[col] = safe_numeric(topsis_criteria[col])

# Ensemble tahminlerinden doktor/yatak/hastane vb. kolonları bilgi amaçlı ekliyoruz.
ensemble_cols = ["il_key", "yıl", "doktor", "ebe", "diger", "hemsire", "yatak", "hastane"]
available_ensemble_cols = [c for c in ensemble_cols if c in ensemble.columns]

future = topsis_criteria.merge(
    ensemble[available_ensemble_cols],
    on=["il_key", "yıl"],
    how="left"
)

print("Birleşmiş TOPSIS + ensemble veri:", future.shape)
display(future.head())

Birleşmiş TOPSIS + ensemble veri: (486, 14)


,il_adı,il_key,yıl,nufus,oran_65_plus,oran_0_14,hekim_1000,yatak_100000,doktor,ebe,diger,hemsire,yatak,hastane
0,Adana,adana,2025,2283609,10.586751,21.993213,2.602272,323.985432,5948.0,1735.0,8415.0,6921.0,7460.0,32.0
1,Adana,adana,2026,2293499,11.016234,21.286629,2.631855,329.220907,6082.0,1749.0,8949.0,6921.0,7528.0,33.0
2,Adana,adana,2027,2299727,11.457670,20.825061,2.657828,334.550691,6217.0,1762.0,9479.0,6921.0,7596.0,33.0
3,Adana,adana,2028,2305551,11.919366,20.374964,2.694466,339.729930,6352.0,1776.0,10007.0,6921.0,7663.0,33.0
4,Adana,adana,2029,2310973,12.396739,19.750900,2.736625,344.715297,6487.0,1790.0,10531.0,6921.0,7730.0,34.0


In [10]:
# %% [9] 2025 REFERANSINA GÖRE TEMEL KRİTER AĞIRLIKLARINI HESAPLA
# Mevcut notebook mantığı korunuyor:
# 65+ oranı, 0-14 oranı, hekim oranı, yatak oranı için Shannon Entropi ağırlığı hesaplanır.
# Daha sonra bu ağırlıklar bütçe etkisine yer açmak için (1 - BUDGET_EFFECT_WEIGHT) ile ölçeklenir.

ref_df = future[future["yıl"] == 2025].dropna(subset=BASE_CRITERIA_COLS)
base_weights_raw = shannon_entropi_agirlik(ref_df, BASE_CRITERIA_COLS)

# Bütçe etkisi için ana kriterleri ölçekle
dynamic_weights = {
    k: v * (1.0 - BUDGET_EFFECT_WEIGHT)
    for k, v in base_weights_raw.items()
}

# Önceki kümülatif bütçe desteği ayrı bir cost kriteridir
dynamic_weights[BUDGET_SUPPORT_COL] = BUDGET_EFFECT_WEIGHT

dynamic_criteria_types = BASE_CRITERIA_TYPES.copy()
dynamic_criteria_types[BUDGET_SUPPORT_COL] = "cost"

DYNAMIC_CRITERIA_COLS = BASE_CRITERIA_COLS + [BUDGET_SUPPORT_COL]

print("\n" + "="*72)
print("DİNAMİK TOPSIS KRİTER AĞIRLIKLARI")
print("="*72)
print("Ham entropi ağırlıkları:")
for k, v in base_weights_raw.items():
    print(f"  {k:<34} {v:.4f}")

print("\nBütçe etkisi eklendikten sonra kullanılan ağırlıklar:")
for k, v in dynamic_weights.items():
    print(f"  {k:<34} {v:.4f}  | {dynamic_criteria_types[k]}")
print(f"\nToplam ağırlık: {sum(dynamic_weights.values()):.4f}")
print("="*72)


DİNAMİK TOPSIS KRİTER AĞIRLIKLARI
Ham entropi ağırlıkları:
  oran_65_plus                       0.3698
  oran_0_14                          0.1976
  hekim_1000                         0.2205
  yatak_100000                       0.2121

Bütçe etkisi eklendikten sonra kullanılan ağırlıklar:
  oran_65_plus                       0.2774  | benefit
  oran_0_14                          0.1482  | benefit
  hekim_1000                         0.1654  | cost
  yatak_100000                       0.1591  | cost
  onceki_kumulatif_butce_destegi     0.2500  | cost

Toplam ağırlık: 1.0000


In [11]:
# %% [10] BÜTÇE DESTEĞİ KRİTERİNİ HESAPLAYAN FONKSİYON
def add_budget_support_criterion(df_year, city_state):
    """
    Önceki yıllarda alınan kümülatif bütçe desteğini TOPSIS'e ekler.

    BUDGET_SUPPORT_MODE = "per_capita":
        önceki kümülatif bütçe / nüfus

    BUDGET_SUPPORT_MODE = "absolute":
        önceki kümülatif bütçe
    """
    d = df_year.copy()

    d = d.merge(
        city_state[["il_key", "kumulatif_butce"]],
        on="il_key",
        how="left"
    )

    d["kumulatif_butce"] = d["kumulatif_butce"].fillna(0.0)

    if BUDGET_SUPPORT_MODE == "per_capita":
        # Bütçe kolonu milyon birimindeyse kişi başına gerçek birim:
        # milyon * 1_000_000 / nüfus
        d[BUDGET_SUPPORT_COL] = (
            d["kumulatif_butce"] * 1_000_000
        ) / d["nufus"].replace(0, np.nan)

        d[BUDGET_SUPPORT_COL] = d[BUDGET_SUPPORT_COL].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    elif BUDGET_SUPPORT_MODE == "absolute":
        d[BUDGET_SUPPORT_COL] = d["kumulatif_butce"]

    else:
        raise ValueError("BUDGET_SUPPORT_MODE sadece 'per_capita' veya 'absolute' olabilir.")

    return d

In [12]:
# %% [11] DİNAMİK TOPSIS + BÜTÇE DAĞITIM DÖNGÜSÜ
# Bu hücre asıl çözümü üretir.
# Kritik nokta:
#   2025 bütçesi -> 2025 TOPSIS skoruna göre dağıtılır.
#   Dağıtılan 2025 bütçesi -> 2026 TOPSIS hesabında cost kriteri olarak kullanılır.
#   Aynı mantık 2030'a kadar devam eder.

# İl bazlı kümülatif bütçe durumu
city_state = (
    future[future["yıl"] == min(GELECEK_YILLAR)][["il_key", "il_adı"]]
    .drop_duplicates()
    .copy()
)
city_state["kumulatif_butce"] = 0.0

dynamic_results = []

for year in GELECEK_YILLAR:
    df_year = future[future["yıl"] == year].copy()

    # Önceki yıllarda alınan bütçe desteğini TOPSIS kriteri olarak ekle
    df_year = add_budget_support_criterion(df_year, city_state)

    # O yıl için TOPSIS hesapla
    result_year = calculate_topsis_for_one_year(
        df_year=df_year,
        criteria_cols=DYNAMIC_CRITERIA_COLS,
        weights=dynamic_weights,
        criteria_types=dynamic_criteria_types
    )

    # TOPSIS skorundan bütçe dağıtım payı üret
    score_for_allocation = np.clip(result_year["topsis_skor"].values, 0, None)

    if BUDGET_ALLOCATION_GAMMA != 1.0:
        score_for_allocation = score_for_allocation ** BUDGET_ALLOCATION_GAMMA

    result_year["butce_dagitim_payi"] = normalize_to_sum_one(score_for_allocation)

    # O yılın toplam bütçesini al
    budget_match = budget_df.loc[budget_df["yıl"] == year, BUDGET_VALUE_COL]

    if budget_match.empty:
        raise ValueError(f"{year} yılı için bütçe değeri bulunamadı.")

    total_budget = float(budget_match.iloc[0])

    # Bütçeyi dağıt
    result_year["yil_toplam_butce"] = total_budget
    result_year["ayrilan_butce"] = result_year["butce_dagitim_payi"] * total_budget

    # Kişi başına ayrılan bütçe
    result_year["ayrilan_butce_kisi_basi"] = (
        result_year["ayrilan_butce"] * 1_000_000
    ) / result_year["nufus"].replace(0, np.nan)

    result_year["ayrilan_butce_kisi_basi"] = (
        result_year["ayrilan_butce_kisi_basi"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    # Yıl sonucunu sakla
    dynamic_results.append(result_year)

    # Bu yıl verilen bütçeyi sonraki yıl için kümülatif duruma ekle
    update_budget = result_year[["il_key", "ayrilan_butce"]].copy()

    city_state = city_state.merge(update_budget, on="il_key", how="left")
    city_state["ayrilan_butce"] = city_state["ayrilan_butce"].fillna(0.0)
    city_state["kumulatif_butce"] = city_state["kumulatif_butce"] + city_state["ayrilan_butce"]
    city_state = city_state.drop(columns=["ayrilan_butce"])

dynamic_all = pd.concat(dynamic_results, ignore_index=True)
dynamic_all = dynamic_all.sort_values(["yıl", "topsis_sira"]).reset_index(drop=True)

print("Dinamik TOPSIS + bütçe dağıtımı tamamlandı:", dynamic_all.shape)

# Kontrol: Her yıl dağıtılan bütçe toplamı, bütçe dosyasındaki toplamla aynı olmalı.
budget_check = (
    dynamic_all.groupby("yıl", as_index=False)["ayrilan_butce"]
    .sum()
    .merge(budget_df[["yıl", BUDGET_VALUE_COL]], on="yıl", how="left")
)
budget_check["fark"] = budget_check["ayrilan_butce"] - budget_check[BUDGET_VALUE_COL]

print("\nBütçe toplam kontrolü:")
display(budget_check)

Dinamik TOPSIS + bütçe dağıtımı tamamlandı: (486, 35)

Bütçe toplam kontrolü:


,yıl,ayrilan_butce,butce_TL_turetilen (Milyon TL),fark
0,2025,2935383.62,2935383.62,0.000000e+00
1,2026,3585538.43,3585538.43,0.000000e+00
2,2027,4456543.41,4456543.41,-9.313226e-10
3,2028,5437758.39,5437758.39,0.000000e+00
4,2029,7085975.58,7085975.58,0.000000e+00
5,2030,8512009.65,8512009.65,0.000000e+00


In [13]:
# %% [12] ÇIKTI TABLOLARINI HAZIRLA
# Ana detay tablo
detail_cols = [
    "il_adı", "il_key", "yıl", "nufus",
    "oran_65_plus", "oran_0_14", "hekim_1000", "yatak_100000",
    BUDGET_SUPPORT_COL,
    "kumulatif_butce",
    "topsis_skor", "topsis_sira", "oncelik_grubu",
    "butce_dagitim_payi",
    "yil_toplam_butce",
    "ayrilan_butce",
    "ayrilan_butce_kisi_basi",
    "ideal_iyi_uzaklik", "ideal_kotu_uzaklik",
]

# Model tahmin kolonlarını da ekle
for col in ["doktor", "ebe", "diger", "hemsire", "yatak", "hastane"]:
    if col in dynamic_all.columns:
        detail_cols.append(col)

# Normalizasyon/ağırlıklı kolonları da ekle
for col in DYNAMIC_CRITERIA_COLS:
    for suffix in ["_norm", "_agirlikli"]:
        c = f"{col}{suffix}"
        if c in dynamic_all.columns:
            detail_cols.append(c)

detail_out = dynamic_all[detail_cols].copy()

# Özet tablo: yıl bazında ilk 20
top20_out = (
    detail_out.sort_values(["yıl", "topsis_sira"])
    .groupby("yıl")
    .head(20)
    .reset_index(drop=True)
)

# Kriter ağırlıkları tablosu
weights_out = pd.DataFrame({
    "kriter": list(dynamic_weights.keys()),
    "agirlik": list(dynamic_weights.values()),
    "tip": [dynamic_criteria_types[k] for k in dynamic_weights.keys()],
    "aciklama": [
        "65+ nüfus oranı yüksekse sağlık ihtiyacı artar",
        "0-14 nüfus oranı yüksekse sağlık ihtiyacı artar",
        "1000 kişiye düşen hekim yüksekse ihtiyaç azalır",
        "100 bin kişiye düşen yatak yüksekse ihtiyaç azalır",
        "Önceki yıllarda alınan kümülatif bütçe desteği yüksekse yeni ihtiyaç azalır",
    ]
})

display(detail_out.head())
display(top20_out[["yıl", "topsis_sira", "il_adı", "topsis_skor", "butce_dagitim_payi", "ayrilan_butce"]].head(30))
display(weights_out)

,il_adı,il_key,yıl,nufus,oran_65_plus,oran_0_14,hekim_1000,yatak_100000,onceki_kumulatif_butce_destegi,kumulatif_butce,...,oran_65_plus_norm,oran_65_plus_agirlikli,oran_0_14_norm,oran_0_14_agirlikli,hekim_1000_norm,hekim_1000_agirlikli,yatak_100000_norm,yatak_100000_agirlikli,onceki_kumulatif_butce_destegi_norm,onceki_kumulatif_butce_destegi_agirlikli
0,Sinop,sinop,2025,225848,21.655273,14.663845,1.582496,299.705865,0.0,0.0,...,0.178569,0.049533,0.078753,0.011668,0.078628,0.013002,0.105686,0.016813,0.0,0.0
1,Artvin,artvin,2025,167531,19.248975,14.686237,1.818950,226.761346,0.0,0.0,...,0.158727,0.044029,0.078874,0.011686,0.090376,0.014945,0.079963,0.012721,0.0,0.0
2,Kastamonu,kastamonu,2025,379934,21.084978,14.383551,1.868085,318.529482,0.0,0.0,...,0.173866,0.048228,0.077248,0.011445,0.092818,0.015348,0.112324,0.017869,0.0,0.0
3,Çankırı,cankiri,2025,200549,18.176605,16.054929,1.753825,238.959278,0.0,0.0,...,0.149884,0.041576,0.086224,0.012775,0.087141,0.014410,0.084265,0.013405,0.0,0.0
4,Bartın,bartin,2025,206663,17.401760,14.282673,1.673399,251.487057,0.0,0.0,...,0.143494,0.039804,0.076706,0.011365,0.083144,0.013749,0.088682,0.014108,0.0,0.0


,yıl,topsis_sira,il_adı,topsis_skor,butce_dagitim_payi,ayrilan_butce
0,2025,1,Sinop,0.711775,0.016478,48369.168148
1,2025,2,Artvin,0.697187,0.016140,47377.846094
2,2025,3,Kastamonu,0.689018,0.015951,46822.691466
3,2025,4,Çankırı,0.688731,0.015945,46803.230308
4,2025,5,Bartın,0.660507,0.015291,44885.230178
5,2025,6,Balıkesir,0.657129,0.015213,44655.658570
6,2025,7,Amasya,0.655155,0.015167,44521.522065
7,2025,8,Burdur,0.648499,0.015013,44069.237959
8,2025,9,Çorum,0.644949,0.014931,43827.989318
9,2025,10,Ordu,0.644717,0.014926,43812.243372


,kriter,agirlik,tip,aciklama
0,oran_65_plus,0.277387,benefit,65+ nüfus oranı yüksekse sağlık ihtiyacı artar
1,oran_0_14,0.148165,benefit,0-14 nüfus oranı yüksekse sağlık ihtiyacı artar
2,hekim_1000,0.165361,cost,1000 kişiye düşen hekim yüksekse ihtiyaç azalır
3,yatak_100000,0.159088,cost,100 bin kişiye düşen yatak yüksekse ihtiyaç az...
4,onceki_kumulatif_butce_destegi,0.250000,cost,Önceki yıllarda alınan kümülatif bütçe desteği...


In [14]:
# %% [13] ÇIKTILARI KAYDET
detail_csv  = "09_topsis_dinamik_butce_dagitim_2025_2030.csv"
top20_csv   = "09_topsis_dinamik_butce_dagitim_ilk20_2025_2030.csv"
weights_csv = "09_topsis_dinamik_kriter_agirliklari.csv"
excel_out   = "09_topsis_dinamik_tek_dosya_2025_2030.xlsx"

detail_out.to_csv(detail_csv, index=False, encoding="utf-8-sig")
top20_out.to_csv(top20_csv, index=False, encoding="utf-8-sig")
weights_out.to_csv(weights_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(excel_out) as writer:
    detail_out.to_excel(writer, sheet_name="Dinamik_Dagitim", index=False)
    top20_out.to_excel(writer, sheet_name="Yillik_Ilk20", index=False)
    weights_out.to_excel(writer, sheet_name="Kriter_Agirliklari", index=False)
    budget_check.to_excel(writer, sheet_name="Butce_Kontrol", index=False)

print("\n[OK] Çıktılar kaydedildi:")
print(f"  - {detail_csv}")
print(f"  - {top20_csv}")
print(f"  - {weights_csv}")
print(f"  - {excel_out}")


[OK] Çıktılar kaydedildi:
  - 09_topsis_dinamik_butce_dagitim_2025_2030.csv
  - 09_topsis_dinamik_butce_dagitim_ilk20_2025_2030.csv
  - 09_topsis_dinamik_kriter_agirliklari.csv
  - 09_topsis_dinamik_tek_dosya_2025_2030.xlsx


In [15]:
# %% [14] YILLIK İLK 10 ÖNCELİKLİ İLİ GÖSTER
for year in GELECEK_YILLAR:
    print("\n" + "="*90)
    print(f"{year} YILI - DİNAMİK TOPSIS'E GÖRE İLK 10 İL")
    print("="*90)

    show = detail_out[detail_out["yıl"] == year].head(10)[[
        "topsis_sira",
        "il_adı",
        "topsis_skor",
        BUDGET_SUPPORT_COL,
        "butce_dagitim_payi",
        "ayrilan_butce",
        "ayrilan_butce_kisi_basi",
    ]].copy()

    print(show.round(4).to_string(index=False))


2025 YILI - DİNAMİK TOPSIS'E GÖRE İLK 10 İL
 topsis_sira    il_adı  topsis_skor  onceki_kumulatif_butce_destegi  butce_dagitim_payi  ayrilan_butce  ayrilan_butce_kisi_basi
           1     Sinop       0.7118                             0.0              0.0165     48369.1681              214166.9094
           2    Artvin       0.6972                             0.0              0.0161     47377.8461              282800.4733
           3 Kastamonu       0.6890                             0.0              0.0160     46822.6915              123239.0138
           4   Çankırı       0.6887                             0.0              0.0159     46803.2303              233375.5357
           5    Bartın       0.6605                             0.0              0.0153     44885.2302              217190.4510
           6 Balıkesir       0.6571                             0.0              0.0152     44655.6586               34764.5524
           7    Amasya       0.6552                        

In [16]:
# %% [15] SABİT TOPSIS MANTIĞI İLE DİNAMİK TOPSIS'İ KARŞILAŞTIR
# Bu hücre, eski problem olan "aynı ilin sürekli en üstte kalması" durumunu gözlemlemek için eklenmiştir.
# Sadece yorumlama ve kontrol amaçlıdır.

# Sabit model: budget_support hiç kullanılmaz, sadece temel kriterlerle her yıl TOPSIS hesaplanır.
static_results = []
for year in GELECEK_YILLAR:
    df_year = future[future["yıl"] == year].copy()
    res = calculate_topsis_for_one_year(
        df_year=df_year,
        criteria_cols=BASE_CRITERIA_COLS,
        weights=base_weights_raw,
        criteria_types=BASE_CRITERIA_TYPES
    )
    res["model"] = "static"
    static_results.append(res)

static_all = pd.concat(static_results, ignore_index=True)

compare_rank = (
    static_all[["il_key", "yıl", "il_adı", "topsis_sira", "topsis_skor"]]
    .rename(columns={
        "topsis_sira": "static_sira",
        "topsis_skor": "static_skor"
    })
    .merge(
        detail_out[["il_key", "yıl", "topsis_sira", "topsis_skor", "ayrilan_butce", BUDGET_SUPPORT_COL]]
        .rename(columns={
            "topsis_sira": "dynamic_sira",
            "topsis_skor": "dynamic_skor"
        }),
        on=["il_key", "yıl"],
        how="inner"
    )
)

compare_rank["sira_degisim"] = compare_rank["static_sira"] - compare_rank["dynamic_sira"]

print("Statik vs dinamik sıralama karşılaştırması hazır.")
display(compare_rank.sort_values(["yıl", "dynamic_sira"]).head(30))

compare_rank.to_csv(
    "09_topsis_static_dynamic_sira_karsilastirma.csv",
    index=False,
    encoding="utf-8-sig"
)

Statik vs dinamik sıralama karşılaştırması hazır.


,il_key,yıl,il_adı,static_sira,static_skor,dynamic_sira,dynamic_skor,ayrilan_butce,onceki_kumulatif_butce_destegi,sira_degisim
0,sinop,2025,Sinop,1,0.711775,1,0.711775,48369.168148,0.0,0
1,artvin,2025,Artvin,2,0.697187,2,0.697187,47377.846094,0.0,0
2,kastamonu,2025,Kastamonu,3,0.689018,3,0.689018,46822.691466,0.0,0
3,cankiri,2025,Çankırı,4,0.688731,4,0.688731,46803.230308,0.0,0
4,bartin,2025,Bartın,5,0.660507,5,0.660507,44885.230178,0.0,0
5,balikesir,2025,Balıkesir,6,0.657129,6,0.657129,44655.658570,0.0,0
6,amasya,2025,Amasya,7,0.655155,7,0.655155,44521.522065,0.0,0
7,burdur,2025,Burdur,8,0.648499,8,0.648499,44069.237959,0.0,0
8,corum,2025,Çorum,9,0.644949,9,0.644949,43827.989318,0.0,0
9,ordu,2025,Ordu,10,0.644717,10,0.644717,43812.243372,0.0,0


In [17]:
# %% [16] DUYARLILIK ANALİZİ FONKSİYONU
# Farklı bütçe etki ağırlıkları denenmek istenirse bu fonksiyon kullanılabilir.
# Örnek:
#   sensitivity = run_sensitivity_analysis([0.15, 0.20, 0.25, 0.30, 0.35])
#   display(sensitivity)

def run_dynamic_model_for_weight(budget_effect_weight):
    """Farklı bütçe etki ağırlıkları için kısa özet üretir."""
    local_weights = {
        k: v * (1.0 - budget_effect_weight)
        for k, v in base_weights_raw.items()
    }
    local_weights[BUDGET_SUPPORT_COL] = budget_effect_weight

    local_types = BASE_CRITERIA_TYPES.copy()
    local_types[BUDGET_SUPPORT_COL] = "cost"

    local_city_state = (
        future[future["yıl"] == min(GELECEK_YILLAR)][["il_key", "il_adı"]]
        .drop_duplicates()
        .copy()
    )
    local_city_state["kumulatif_butce"] = 0.0

    rows = []

    for year in GELECEK_YILLAR:
        df_year = future[future["yıl"] == year].copy()
        df_year = add_budget_support_criterion(df_year, local_city_state)

        result_year = calculate_topsis_for_one_year(
            df_year=df_year,
            criteria_cols=DYNAMIC_CRITERIA_COLS,
            weights=local_weights,
            criteria_types=local_types
        )

        score = np.clip(result_year["topsis_skor"].values, 0, None)
        if BUDGET_ALLOCATION_GAMMA != 1.0:
            score = score ** BUDGET_ALLOCATION_GAMMA

        result_year["butce_dagitim_payi"] = normalize_to_sum_one(score)
        total_budget = float(budget_df.loc[budget_df["yıl"] == year, BUDGET_VALUE_COL].iloc[0])
        result_year["ayrilan_butce"] = result_year["butce_dagitim_payi"] * total_budget

        # İlk 5 ili özetle
        top5 = result_year.head(5)["il_adı"].tolist()

        rows.append({
            "budget_effect_weight": budget_effect_weight,
            "yıl": year,
            "ilk_5_il": ", ".join(top5),
            "birinci_il": result_year.iloc[0]["il_adı"],
            "birinci_skor": result_year.iloc[0]["topsis_skor"],
            "birinci_butce": result_year.iloc[0]["ayrilan_butce"],
        })

        # state update
        update_budget = result_year[["il_key", "ayrilan_butce"]].copy()
        local_city_state = local_city_state.merge(update_budget, on="il_key", how="left")
        local_city_state["ayrilan_butce"] = local_city_state["ayrilan_butce"].fillna(0.0)
        local_city_state["kumulatif_butce"] = (
            local_city_state["kumulatif_butce"] + local_city_state["ayrilan_butce"]
        )
        local_city_state = local_city_state.drop(columns=["ayrilan_butce"])

    return pd.DataFrame(rows)

def run_sensitivity_analysis(weight_list):
    frames = [run_dynamic_model_for_weight(w) for w in weight_list]
    return pd.concat(frames, ignore_index=True)

# Örnek kullanım:
# sensitivity = run_sensitivity_analysis([0.15, 0.20, 0.25, 0.30, 0.35])
# display(sensitivity)
# sensitivity.to_csv("09_topsis_budget_weight_duyarlilik.csv", index=False, encoding="utf-8-sig")